In [1]:
import pandas as pd

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

RNG = np.random.default_rng()

CATEGORY_ORDER = ["Not at all", "Slightly", "Moderately", "Very", "Extremely"]
EVENT_LABELS = ["Silent", "Vehicle", "Human", "JackHammer", "Drill", "Piling",
                "AngleGrinding", "Generator", "Reverse", "Miscconstruct", "Siren"]

OUTPUT_ROOT = Path(r"C:\Users\tangu\PycharmProjects\MSc-Project\Final_Upload_Content")


def randomize_categorical(series, categories):
    mask = series.notna()
    series = series.copy()
    series[mask] = RNG.choice(categories, size=mask.sum())
    return series


def randomize_numeric(series):
    mask = series.notna() & np.isfinite(series)
    series = series.copy()
    if mask.sum() == 0:
        return series
    lo, hi = series[mask].min(), series[mask].max()
    series[mask] = RNG.uniform(lo, hi, size=mask.sum())
    return series


def randomize_dba_spla(df, dba_col="dba", spla_col="avg_spla"):
    df = df.copy()
    df[dba_col] = randomize_numeric(df[dba_col])
    df[spla_col] = 10 ** (df[dba_col] / 10)
    return df


def randomize_cityai_loudness(src_base, dst_base, dates, sensors):
    for date in dates:
        for sensor in sensors:
            folder = Path(src_base) / f"date={date}" / f"sensor_id=ics-{sensor}"
            if not folder.exists():
                continue
            out_folder = Path(dst_base) / f"date={date}" / f"sensor_id=ics-{sensor}"
            out_folder.mkdir(parents=True, exist_ok=True)

            for pq_file in folder.glob("*.parquet"):
                df = pd.read_parquet(pq_file)
                df = randomize_dba_spla(df)
                df.to_parquet(out_folder / pq_file.name)


def randomize_cityai_events(src_base, dst_base, dates, sensors):
    for date in dates:
        for sensor in sensors:
            folder = Path(src_base) / f"date={date}" / f"sensor_id=ics-{sensor}"
            if not folder.exists():
                continue
            out_folder = Path(dst_base) / f"date={date}" / f"sensor_id=ics-{sensor}"
            out_folder.mkdir(parents=True, exist_ok=True)

            for pq_file in folder.glob("*.parquet"):
                df = pd.read_parquet(pq_file)
                if "avg_spl_a" in df.columns:
                    df["avg_spl_a"] = randomize_numeric(df["avg_spl_a"])
                if "avg_sharpness" in df.columns:
                    df["avg_sharpness"] = randomize_numeric(df["avg_sharpness"])
                if "label" in df.columns:
                    df["label"] = randomize_categorical(df["label"], EVENT_LABELS)
                df.to_parquet(out_folder / pq_file.name)


def randomize_munisense(src_path, dst_path):
    df = pd.read_csv(src_path, sep=";")
    if "laeq_avg" in df.columns:
        df["laeq_avg"] = randomize_numeric(df["laeq_avg"])
    df.to_csv(dst_path, sep=";", index=False)


def randomize_survey(src_path, dst_path, sheet_name="Sheet1"):
    df = pd.read_excel(src_path, sheet_name=sheet_name)
    for col in df.columns:
        if col.startswith(("Q1", "Q2", "Q3", "Q4", "Q5")):
            df[col] = randomize_categorical(df[col], CATEGORY_ORDER)
    df.to_excel(dst_path, index=False)


dates = pd.date_range(start="2025-06-17", end="2025-10-14").strftime("%Y-%m-%d")
sensors = ["02", "03", "04", "05", "06", "07", "09", "11", "12"]

(OUTPUT_ROOT / "Munisense").mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / "Survey").mkdir(parents=True, exist_ok=True)

randomize_cityai_loudness(
    src_base=r"C:\Users\tangu\Desktop\TU Delft Assignements\Year 2\Thesis\Data\CityAI\data\marts\loudness_per_minute",
    dst_base=OUTPUT_ROOT / "CityAI" / "loudness_per_minute",
    dates=dates, sensors=sensors
)

randomize_cityai_events(
    src_base=r"C:\Users\tangu\Desktop\TU Delft Assignements\Year 2\Thesis\Data\CityAI\data\lake\scsm_sources",
    dst_base=OUTPUT_ROOT / "CityAI" / "scsm_sources",
    dates=dates, sensors=sensors
)

randomize_munisense(
    r"C:\Users\tangu\Desktop\TU Delft Assignements\Year 2\Thesis\Data\Munisense\export15\omval.csv",
    OUTPUT_ROOT / "Munisense" / "omval.csv"
)
randomize_munisense(
    r"C:\Users\tangu\Desktop\TU Delft Assignements\Year 2\Thesis\Data\Munisense\export15\merca.csv",
    OUTPUT_ROOT / "Munisense" / "merca.csv"
)

randomize_survey(
    r"C:\Users\tangu\Desktop\TU Delft Assignements\Year 2\Thesis\Data\Survey\Survey Results Treated.xlsx",
    OUTPUT_ROOT / "Survey" / "Survey_Results_Treated.xlsx"
)